# Generalization study — zero-day & data scarcity

Runs the two experiments from `src/run_generalization_study.py` cell by cell so you can watch the output.

- Kernel: the project `.venv` (needs `ipykernel` and `nbformat` installed).
- Checkpoints in `models/generalization/` are reused when present. The first data-scarcity run for a client trains a FedPer trunk without that client, which takes a few minutes per seed.
- To only look at earlier results, run the setup cell and then **Load saved results**, and skip the run cells.
- The only file this notebook writes is `results/generalization_results.json`, in the **Save** cell.

In [ ]:
import os
import sys
import json

# Run from the repo root regardless of where Jupyter starts.
root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(root, "src")) and os.path.dirname(root) != root:
    root = os.path.dirname(root)
os.chdir(root)
sys.path.insert(0, root)

import pandas as pd
import plotly.graph_objects as go
import torch

import src.run_generalization_study as gs
from src.utils import set_seed

print("cwd:", os.getcwd(), "| torch", torch.__version__)
print("seeds:", gs.SEEDS, "| zero-day targets:", gs.ZERO_DAY_TARGETS, "| scarcity targets:", gs.SCARCITY_TARGETS,
      "| scarcity min steps:", gs.SCARCITY_MIN_STEPS)

## Load saved results (optional — skips re-running)

In [ ]:
with open(gs.GENERALIZATION_RESULTS_PATH, "r", encoding="utf-8") as f:
    saved = json.load(f)
zero_day, scarcity = saved["zero_day"], saved["data_scarcity"]
print(saved.get("metadata"))

## Experiment 1 — Zero-day

In [ ]:
set_seed(42)
zero_day = gs.run_zero_day_experiment()

In [ ]:
rows = []
for target, entry in zero_day["targets"].items():
    for method, s in entry["summary"].items():
        row = {
            "target": target,
            "method": method,
            "unseen_recall": s["mean_unseen_attack_recall"],
            "margin": s["mean_detection_margin"],
            "in_dist_f1": s["in_distribution"]["f1_macro"],
            "in_dist_benign_recall": s["in_distribution"]["benign_recall"],
        }
        row.update({f"{src}_benign_flagged": v for src, v in s["foreign_benign_flag_rate"].items()})
        rows.append(row)
pd.DataFrame(rows).round(3)

In [ ]:
# Per-attack recall for one target
target = "mqtt"
entry = zero_day["targets"][target]
pd.DataFrame({
    m: {a: v["mean"] for a, v in s["per_attack"].items()}
    for m, s in entry["summary"].items()
}).round(2)

## Experiment 2 — Data scarcity

Edit `targets` to run a subset (e.g. `["mqtt"]`).

In [ ]:
targets = gs.SCARCITY_TARGETS
scarcity = {"targets": {}}
for t in targets:
    print(f"\n===== Data scarcity: target={t} =====")
    scarcity["targets"][t] = gs.run_data_scarcity_experiment(target=t)

In [ ]:
rows = []
for t, res in scarcity["targets"].items():
    for arm, points in res["summary"].items():
        for p in points:
            rows.append({"target": t, "method": arm, "n": p["n"], "n_train": p["n_train"],
                         "f1_mean": p["f1_macro_mean"], "f1_std": p["f1_macro_std"], "f1_min": p["f1_macro_min"],
                         "benign_recall": p["benign_recall_mean"], "attack_recall": p["attack_recall_mean"]})
df_scarcity = pd.DataFrame(rows)
df_scarcity.pivot_table(index=["target", "n_train"], columns="method", values="f1_mean").round(3)

In [ ]:
for t, res in scarcity["targets"].items():
    fig = go.Figure()
    for arm, points in res["summary"].items():
        fig.add_trace(go.Scatter(
            name=arm, mode="lines+markers",
            x=[p["n_train"] for p in points],
            y=[p["f1_macro_mean"] for p in points],
            error_y=dict(type="data", array=[p["f1_macro_std"] for p in points]),
        ))
    fig.update_layout(title=f"{t}: macro F1 vs N", xaxis_type="log",
                      xaxis_title="N (training samples)", yaxis_title="Macro F1", yaxis_range=[0, 1.05])
    fig.show()

## Save

Writes `results/generalization_results.json` in the same format as the script's `__main__`. `study_results.json` is not touched.

In [ ]:
output = {
    "metadata": {
        "generated": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
        "torch_version": torch.__version__,
        "data_dir": gs.DATA_DIR,
        "runner": "notebooks/generalization_study.ipynb",
    },
    "zero_day": zero_day,
    "data_scarcity": scarcity,
}
with open(gs.GENERALIZATION_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2)
print("Saved", gs.GENERALIZATION_RESULTS_PATH)